# 01. 현재 S&P 500 및 10년 OHLCV 수집
API를 호출하는 유일한 노트북입니다. 최초 수집 또는 명시적인 재수집 때만 실행합니다.

In [4]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path('/Users/choedasom/lab_middle_project')
assert (PROJECT_ROOT / 'src' / 'collect_prices.py').is_file(), f'경로 확인 필요: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

# .env(git에 커밋되지 않음)에서 TIINGO_API_KEY 등을 읽어 환경 변수로 등록한다.
# collect_prices.py의 TIINGO_API_KEY는 모듈 임포트 시점에 os.environ을 읽으므로,
# 반드시 아래 sys.modules 캐시 삭제(재임포트) 이전에 환경 변수를 먼저 설정해야 한다.
env_path = PROJECT_ROOT / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, _, value = line.partition('=')
            os.environ.setdefault(key.strip(), value.strip())

for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]

from src.collect_prices import END_DATE, START_DATE, make_df
from src.get_tickers import get_sp500_universe
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'수집 구간: {START_DATE} ~ {END_DATE} (끝 미포함)')
print(f'TIINGO_API_KEY 설정됨: {bool(os.environ.get("TIINGO_API_KEY"))}')

PROJECT_ROOT: /Users/choedasom/lab_middle_project
수집 구간: 2016-01-01 ~ 2026-07-01 (끝 미포함)
TIINGO_API_KEY 설정됨: True


In [5]:
from src.get_tickers import get_sp500_universe, get_historical_sp500_universe

current_universe = get_sp500_universe()
historical_universe = get_historical_sp500_universe(START_DATE, END_DATE)

# 생존자 편향 보정: "오늘 기준" 종목(503개)뿐 아니라, 분석 기간(START_DATE~END_DATE) 중
# 편출된 종목(파산/상장폐지/인수합병 등)도 수집 대상에 포함한다. 이렇게 안 하면
# "지금까지 살아남은 종목"만으로 과거를 분석하게 되어 결과가 실제보다 좋게 보이는
# 착시(survivorship bias)가 생긴다. 회사명/섹터는 현재 유니버스에서 가져오고,
# 과거에만 존재했던 종목은 그 정보가 없어 Unknown으로 채운다 (수집/분석엔 지장 없음).
sp500_universe = historical_universe.merge(
    current_universe[['ticker', 'company', 'sector']], on='ticker', how='left'
)
sp500_universe['company'] = sp500_universe['company'].fillna(sp500_universe['ticker'])
sp500_universe['sector'] = sp500_universe['sector'].fillna('Unknown')

snp500_tickers = sp500_universe['ticker'].drop_duplicates().tolist()
print(f'수집 대상: {len(snp500_tickers)}종목 (현재 {len(current_universe)}개 + 과거 편출 {len(snp500_tickers) - len(current_universe)}개)')
sp500_universe.head()

[캐시] 구성종목 503건 재사용 (/Users/choedasom/lab_middle_project/data/raw/cache/sp500_universe.csv)
[캐시] 편입/편출 이력 1259건 재사용 (/Users/choedasom/lab_middle_project/data/raw/cache/sp500_membership_history.csv)
[INFO] 2016-01-01~2026-07-01 기간 중 한 번이라도 구성종목이었던 티커 729건
수집 대상: 729종목 (현재 503개 + 과거 편출 226개)


,ticker_original,ticker,company,sector
0,A,A,Agilent Technologies,Health Care
1,AABA,AABA,AABA,Unknown
2,AAL,AAL,AAL,Unknown
3,AAP,AAP,AAP,Unknown
4,AAPL,AAPL,Apple Inc.,Information Technology


In [6]:
# 오래 걸리는 셀입니다. yfinance -> Yahoo chart -> Tiingo 순으로 복구합니다.
final_df_raw, final_missing_df = make_df(snp500_tickers, start=START_DATE, end=END_DATE, universe=sp500_universe)
print(final_df_raw.shape, final_missing_df.shape)

1차 yfinance:   0%|          | 0/729 [00:00<?, ?it/s]

2차 Yahoo chart:   0%|          | 0/124 [00:00<?, ?it/s]

(1663038, 8) (21, 7)


In [7]:
final_df_raw.to_parquet(RAW_DIR / 'final_df_raw.parquet', index=False)
final_missing_df.to_csv(RAW_DIR / 'final_missing_df.csv', index=False)
sp500_universe.to_csv(RAW_DIR / 'sp500_universe.csv', index=False)
print(f'raw 저장 완료: {RAW_DIR}')

raw 저장 완료: /Users/choedasom/lab_middle_project/data/raw


In [8]:
from src.collect_prices import collect_sp500_index

sp500_beta_df = collect_sp500_index(start=START_DATE, end=END_DATE)
sp500_beta_df.to_parquet(RAW_DIR / 'sp500_beta_df.parquet', index=False)
print(f'sp500_beta_df 저장 완료: {sp500_beta_df.shape}, {sp500_beta_df["Date"].min()} ~ {sp500_beta_df["Date"].max()}')

sp500_beta_df 저장 완료: (2637, 2), 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00


In [10]:
final_missing_df

,ticker,company,sector,collected,n_rows,fail_stage,fail_reason
0,ADS,ADS,Unknown,False,0,yahoo+chart+tiingo,yahoo: empty response | chart: HTTP 404 | tiin...
1,BBBY,BBBY,Unknown,False,0,yahoo+chart+tiingo,yahoo: empty response | chart: HTTP 400 | tiin...
2,BK,BK,Unknown,False,0,yahoo+chart,yahoo: empty response | chart: HTTP 404
3,CBS,CBS,Unknown,False,0,yahoo+chart+tiingo,yahoo: empty response | chart: HTTP 404 | tiin...
4,CCE,CCE,Unknown,False,0,yahoo+chart+tiingo,yahoo: empty response | chart: no data | tiing...
5,COG,COG,Unknown,False,0,yahoo+chart+tiingo,yahoo: empty response | chart: HTTP 404 | tiin...
6,CSRA,CSRA,Unknown,False,0,yahoo+chart+tiingo,yahoo: empty response | chart: HTTP 400 | tiin...
7,CTL,CTL,Unknown,False,0,yahoo+chart+tiingo,yahoo: empty response | chart: HTTP 404 | tiin...
8,ENDP,ENDP,Unknown,False,0,yahoo+chart+tiingo,yahoo: empty response | chart: HTTP 404 | tiin...
9,EQR,Equity Residential,Real Estate,False,0,yahoo+chart+tiingo,yahoo: empty response | chart: HTTP 400 | tiin...


In [9]:
sp500_universe

,ticker_original,ticker,company,sector
0,A,A,Agilent Technologies,Health Care
1,AABA,AABA,AABA,Unknown
2,AAL,AAL,AAL,Unknown
3,AAP,AAP,AAP,Unknown
4,AAPL,AAPL,Apple Inc.,Information Technology
...,...,...,...,...
724,YUM,YUM,Yum! Brands,Consumer Discretionary
725,ZBH,ZBH,Zimmer Biomet,Health Care
726,ZBRA,ZBRA,Zebra Technologies,Information Technology
727,ZION,ZION,ZION,Unknown


---

# ⚠️ 이 노트북의 저장된 출력은 2026-08-31 실행분입니다 (2026-09-03 기준)

아래 셀들의 출력은 **708종목 수집** 시점 기록입니다. 이후 `data/raw/final_df_raw.parquet`가
**711종목**으로 보강됐는데, 그 작업을 이 노트북이 아니라 별도로 수행했기 때문입니다
(01번은 네트워크를 쓰는 유일한 노트북이라 700여 종목 전체 재수집을 피했습니다).

## 무엇이 추가됐나 (2026-09-03)

사명·티커가 바뀌어 옛 심볼로는 조회되지 않던 3종목입니다.

<table fit-page-width="true" header-row="true">
<tr><td>**구 티커**</td><td>**신 티커**</td><td>**근거**</td></tr>
<tr><td>ADS</td><td>BFH</td><td>CIK 1101215, 과거명 ALLIANCE DATA SYSTEMS CORP (~2022-03-23)</td></tr>
<tr><td>EQR</td><td>VMRK</td><td>CIK 906107, 과거명 EQUITY RESIDENTIAL (~2026-08-12)</td></tr>
<tr><td>FRC</td><td>FRCB</td><td>동일 법인, NYSE 폐지 후 핑크시트 이전 (2023-05-01 FDIC 관리)</td></tr>
</table>

`data/raw/sp500_universe.csv`도 함께 갱신했습니다 — `TRUSTED_RENAMES`를 적용하면서 구/신
티커를 이중 계상하던 6쌍(BK/BNY, CTL/LUMN, JEC/J, MMC/MRSH, PEAK/DOC, TMK/GL)이 병합돼
**729 → 723종목**으로 정정됐습니다.

## 지금 이 노트북을 처음부터 다시 돌리면 어떻게 되나

**현재 상태가 그대로 재현됩니다.** 추가 조치가 필요 없습니다.

`_load_membership_history()`가 `TRUSTED_RENAMES`를 적용하므로, 편입/편출 이력의 `ADS`·`EQR`·`FRC`가
각각 `BFH`·`VMRK`·`FRCB`로 정규화된 채 `get_historical_sp500_universe()`에 전달되고,
수집기가 신 티커로 가격을 받아옵니다. 즉 **코드는 이미 재현 가능한 상태**이고, 낡은 것은
아래 셀들의 저장된 출력뿐입니다.

검증 절차(야후 검색 → SEC CIK/formerNames → 가격대 대조)와 복구/제외 판단 근거는
`src/get_tickers.py`의 `TRUSTED_RENAMES` 주석에 전부 기록돼 있습니다.
